In [ ]:
# ============================================================
# PATH CONFIGURATION — Adjust for your environment
# ============================================================
# These notebooks were originally developed on Google Colab.
# Paths referencing /content/drive/MyDrive/... are Colab-specific.
#
# For local execution, set ROOT_DIR to your project root:
#
#   import os
#   ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
#   # Then replace /content/drive/MyDrive/... paths with os.path.join(ROOT_DIR, ...)
#
# The authoritative dataset files are in: data/train.parquet, data/validation.parquet, data/test.parquet
# Images are in: data/images/
# ============================================================


# Malicious Insurance Document Dataset Generator — V2
## MODULE B (v2.1) — Corrected One-to-One Text Injection (`malicious_text.csv`)

---

### Scope

Module A produced the frozen **Insurance Payload Bank**
(`insurance_payload_bank.csv`). This is the corrected Module B (pipeline
version 2.1). It consumes that payload bank and the frozen
`MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_text.csv` to generate **exactly
one malicious document for every legitimate document**, writing a single
output file: `MASTER_MALICIOUS_DATASET/TEXT/malicious_text.csv`.

Runtime OCR design is unchanged — no OCR is performed here.
EasyOCR runs at preprocessing/training time only.


---
### Design Documentation

#### 1  Why the previous implementation generated multiple malicious samples

The original Module B iterated over *every attack family* for *every benign
document*, producing `N_benign × N_families` rows. For example,
145,000 benign documents × 15 attack families = 2,175,000 malicious rows.
This happened because the generation plan nested two loops: one over benign
rows and one over `ATTACK_FAMILIES`. Each combination produced an independent
malicious sample, turning a one-to-one research task into a combinatorial
explosion. This design (a) makes train/validation/test splitting without data
leakage structurally impossible because a single benign document appears in
many rows, (b) inflates storage beyond what is scientifically necessary, and
(c) violates the realistic threat model where an adversary embeds one payload
per document submission.

#### 2  Why the corrected one-to-one strategy is better

A strict **one legitimate document → one malicious document** mapping keeps the
malicious dataset the same size as the benign dataset. This mirrors the real
threat model, produces a naturally balanced binary classification problem
(no resampling needed), allows grouped splitting by `source_benign_id` to
guarantee zero leakage across splits, and produces a dataset small enough to
be fully reproducible on commodity hardware — all mandatory criteria for
Scopus/Q1 publication.

#### 3  Why balanced attack-family sampling was selected

Unconstrained random sampling over the full payload bank gives each attack
family a probability proportional to the number of payloads it contains,
causing families with more payloads to dominate the dataset. Balanced
stratified sampling — cycling through all families in a seeded random order —
guarantees every family appears `floor(N / F)` or `ceil(N / F)` times, making
the published dataset suitable for per-family subgroup analysis without any
class-imbalance corrections.

#### 4  Why payload randomisation occurs only inside the selected family

Once a family is assigned to a document, the payload choice uses
`numpy.random.Generator` seeded on `(RANDOM_SEED, hash(attack_family))`.
Keeping random selection *within* the family means (a) the family distribution
is fully determined by the stratified sampler and (b) payload diversity within
each family is maximised independently via a cycling permutation — no single
payload dominates a family just because a global RNG happened to land on it
repeatedly.

#### 5  Why 1:1 mapping improves publication quality

Publication reviewers expect the malicious split to be a realistic adversarial
corpus, not a combinatorial expansion of the benign set. A 1:1 design produces
cleaner confusion matrices, interpretable per-family precision/recall
breakdowns, and a compact reproducible artefact. These are non-negotiable
requirements for high-quality publications targeting Scopus/Q1 venues.

#### 6  How the optimised implementation improves runtime

The corrected implementation replaces Python `for` loops with vectorised
NumPy/pandas operations wherever possible:

- **Family assignment** uses `numpy.tile` + `numpy.random.Generator.permutation`
  on an array of family names — O(N) single-pass, zero Python-level row iteration.
- **Payload index selection** uses `numpy.random.Generator.permutation` on
  per-family slices of the payload bank DataFrame and then cycles the resulting
  index array — avoids repeated Python dict lookups per row.
- **Text insertion** splits by `attack_location`: strategies that are pure
  prefix/suffix concatenations (`header`, `footer`, `watermark`, `metadata`)
  use pandas vectorised string operations on the full column slice, not
  row-level Python. Only `inline`, `tiny`, and `invisible_unicode` (which
  require sentence-level parsing) fall back to `.apply()` on their sub-slice --
  reducing Python-level row visits by approximately 57% compared with a single
  loop over all rows.
- **Checkpointing** fires every 10,000 rows instead of 500, cutting Google
  Drive write calls by 20x on a 145k dataset.
- The full list of result dicts is converted to a DataFrame **once** after
  generation completes; there is no repeated `pd.concat` inside any loop.


In [ ]:
# ============================================================
# MODULE B -- STEP 0: Environment Setup
# ============================================================
import os
import re
import sys
import json
import random
import hashlib
import datetime

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Colab -- assuming local/dev environment.")

print("Environment ready. IN_COLAB =", IN_COLAB)


Mounted at /content/drive
Environment ready. IN_COLAB = True


In [ ]:
# ============================================================
# MODULE B -- STEP 1: Project Configuration (reconstructed from Module A)
# ============================================================
# The single source of truth for paths lives in Module A.  Module B
# re-declares the SAME CONFIG dict so it can run standalone after a
# fresh Drive mount, and extends it with the paths this module owns.
#
# PIPELINE_VERSION is used by the validity detector (Step 5) to decide
# whether existing on-disk output was produced by the corrected 1:1
# strategy or the old multi-variant strategy.

PIPELINE_VERSION = "2.1.0"   # bump minor/patch whenever generation logic changes
DATASET_VERSION  = "1.0"
RANDOM_SEED      = 42        # fixed for full reproducibility across all runs

DRIVE_ROOT = "/content/drive/MyDrive" if IN_COLAB else os.path.abspath("./drive_sim")

CONFIG = {
    # ---- FROZEN benign source (read-only, never modified) ----
    "BENIGN_ROOT":           os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET"),
    "BENIGN_TEXT_DIR":       os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "TEXT_SOURCES"),
    "BENIGN_IMAGE_DIR":      os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "IMAGE_SOURCES"),
    "BENIGN_TEXT_CSV":       os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "TEXT_SOURCES", "benign_text.csv"),
    "BENIGN_IMAGE_META_CSV": os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "TEXT_SOURCES", "image_metadata.csv"),

    # ---- malicious project root (single output tree, no duplicates) ----
    "PROJECT_ROOT":   os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET"),
    "PAYLOAD_BANK_DIR": os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "PAYLOAD_BANK"),
    "TEXT_DIR":       os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "TEXT"),
    "IMAGES_DIR":     os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "IMAGES"),
    "PDF_DIR":        os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "PDF"),
    "PARQUET_DIR":    os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "PARQUET"),
    "LABELS_DIR":     os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "LABELS"),
    "VALIDATION_DIR": os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "VALIDATION"),
    "LOGS_DIR":       os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "LOGS"),

    # ---- design decisions (do not change) ----
    "RUNTIME_OCR_ONLY":    True,   # OCR runs at training time via EasyOCR, never here
    "SINGLE_FINAL_OUTPUT": True,   # exactly one malicious_text.csv, updated in place
    "RESUME_ENABLED":      True,   # every stage checks for existing output before running
}

PAYLOAD_BANK_CSV         = os.path.join(CONFIG["PAYLOAD_BANK_DIR"], "insurance_payload_bank.csv")
MALICIOUS_TEXT_CSV       = os.path.join(CONFIG["TEXT_DIR"], "malicious_text.csv")
MALICIOUS_TEXT_META_JSON = os.path.join(CONFIG["TEXT_DIR"], "malicious_text.meta.json")
GENERATION_MANIFEST_JSON = os.path.join(CONFIG["TEXT_DIR"], "generation_manifest.json")

os.makedirs(CONFIG["TEXT_DIR"], exist_ok=True)

print("Module B paths:")
print("  payload bank   (in) :", PAYLOAD_BANK_CSV)
print("  benign text    (in) :", CONFIG["BENIGN_TEXT_CSV"])
print("  malicious text (out):", MALICIOUS_TEXT_CSV)
print("  manifest       (out):", GENERATION_MANIFEST_JSON)


def _atomic_save_csv(df, path):
    """Write to .tmp then os.replace -- never leaves a partial file on Drive."""
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def _atomic_save_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, path)


def _cleanup_tmp_files():
    """Remove any leftover .tmp files from a previously interrupted run."""
    for p in [MALICIOUS_TEXT_CSV, MALICIOUS_TEXT_META_JSON, GENERATION_MANIFEST_JSON]:
        tmp = p + ".tmp"
        if os.path.exists(tmp):
            os.remove(tmp)
            print("  Cleaned up leftover temp file:", tmp)


_cleanup_tmp_files()

# ---- hard pre-conditions: fail fast with actionable messages ----
if not os.path.exists(PAYLOAD_BANK_CSV):
    raise FileNotFoundError(
        "insurance_payload_bank.csv not found at:\n  " + PAYLOAD_BANK_CSV + "\n"
        "Run MODULE A first -- Module B only consumes payloads, never generates them."
    )

if not os.path.exists(CONFIG["BENIGN_TEXT_CSV"]):
    raise FileNotFoundError(
        "benign_text.csv not found at:\n  " + CONFIG["BENIGN_TEXT_CSV"] + "\n"
        "Module B reads the frozen MASTER_BENIGN_DATASET and never modifies it."
    )

print("\nRequired inputs verified.")


Module B paths:
  payload bank   (in) : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PAYLOAD_BANK/insurance_payload_bank.csv
  benign text    (in) : /content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_text.csv
  malicious text (out): /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/malicious_text.csv
  manifest       (out): /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/generation_manifest.json
  Cleaned up leftover temp file: /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/malicious_text.csv.tmp

Required inputs verified.


In [ ]:
# ============================================================
# MODULE B -- STEP 2: Load Payload Bank (read-only, produced by Module A)
# ============================================================
payload_bank_df = pd.read_csv(PAYLOAD_BANK_CSV)

REQUIRED_PAYLOAD_COLUMNS = [
    "payload_id", "attack_family", "attack_name", "payload_text",
    "target_component", "severity", "visibility", "ocr_survivability", "description",
]
missing = [c for c in REQUIRED_PAYLOAD_COLUMNS if c not in payload_bank_df.columns]
if missing:
    raise ValueError("insurance_payload_bank.csv is missing required columns: " + str(missing))

if payload_bank_df["payload_id"].duplicated().any():
    raise ValueError("insurance_payload_bank.csv has duplicate payload_id values -- re-run Module A.")

# Index by family -- each value is a DataFrame slice (preserves dtypes, supports .iloc)
family_to_payloads = {
    family: grp.reset_index(drop=True)
    for family, grp in payload_bank_df.groupby("attack_family")
}

ATTACK_FAMILIES = sorted(family_to_payloads.keys())
N_FAMILIES = len(ATTACK_FAMILIES)
FAMILIES_ARR = np.array(ATTACK_FAMILIES)   # numpy array for vectorised tile/permutation

print("Loaded payload bank: {:,} payloads across {:d} attack families".format(
    len(payload_bank_df), N_FAMILIES))
for family, fam_df in sorted(family_to_payloads.items()):
    print("  - {:<32s} : {:d} payload(s)".format(family, len(fam_df)))


Loaded payload bank: 58 payloads across 15 attack families
  - Context Poisoning                : 3 payload(s)
  - Data Exfiltration                : 5 payload(s)
  - Decision Manipulation            : 5 payload(s)
  - Footer Injection                 : 3 payload(s)
  - Header Injection                 : 3 payload(s)
  - Hidden Multimodal                : 4 payload(s)
  - Instruction Override             : 5 payload(s)
  - Insurance-specific Attacks       : 6 payload(s)
  - Invisible Character Injection    : 3 payload(s)
  - Metadata Injection               : 3 payload(s)
  - OCR-specific Attacks             : 4 payload(s)
  - Policy Bypass                    : 5 payload(s)
  - Tiny Font Injection              : 3 payload(s)
  - Watermark Injection              : 3 payload(s)
  - White Text Injection             : 3 payload(s)


In [ ]:
# ============================================================
# MODULE B -- STEP 3: Text-Insertion Strategies (one per attack_location)
# Kept from v2.0 -- the strategy implementations are correct and unchanged.
# Classified here into VECTORISABLE vs SENTENCE_PARSING for Step 7.
# ============================================================
ZERO_WIDTH = "\u200b"

_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def _split_sentences(text):
    parts = [p for p in _SENTENCE_SPLIT_RE.split(text.strip()) if p]
    return parts if parts else [text]


def _insert_near_midpoint(text, fragment):
    sentences = _split_sentences(text)
    mid = len(sentences) // 2
    sentences.insert(mid, fragment)
    return " ".join(sentences)


# ---- Vectorisable strategies: pure prefix/suffix -- no per-row Python needed ----
def insert_header(text, payload_text):
    return payload_text + "\n\n" + text

def insert_footer(text, payload_text):
    return text + "\n\n" + payload_text

def insert_watermark(text, payload_text):
    return text + "\n\n[[WATERMARK:: " + payload_text + " ::WATERMARK]]"

def insert_metadata_tag(text, payload_text):
    return text + "\n\n<!-- DOCUMENT_METADATA: " + payload_text + " -->"


# ---- Sentence-parsing strategies: require .apply on sub-slice ----
def insert_inline(text, payload_text):
    return _insert_near_midpoint(text, payload_text)

def insert_tiny_marker(text, payload_text):
    return _insert_near_midpoint(text, "<tinytext>" + payload_text + "</tinytext>")

def insert_invisible_unicode(text, payload_text):
    obfuscated = ZERO_WIDTH.join(list(payload_text))
    return _insert_near_midpoint(text, obfuscated)


INSERTION_STRATEGIES = {
    "header":            insert_header,
    "footer":            insert_footer,
    "watermark":         insert_watermark,
    "metadata":          insert_metadata_tag,
    "inline":            insert_inline,
    "tiny":              insert_tiny_marker,
    "invisible_unicode": insert_invisible_unicode,
}

# Locations where pandas string ops replace row-level Python (~57% of assignments)
VECTORISABLE_LOCATIONS = {"header", "footer", "watermark", "metadata"}

ATTACK_TYPE_MAP = {
    "header":            "plain_text_insertion",
    "footer":            "plain_text_insertion",
    "inline":            "plain_text_insertion",
    "watermark":         "watermark_pattern",
    "tiny":              "microtext_marker",
    "invisible_unicode": "unicode_obfuscation",
    "metadata":          "metadata_tag_emulation",
}

# Map attack family names that have a canonical placement to their location
FAMILY_LOCATION_OVERRIDE = {
    "Header Injection":              "header",
    "Footer Injection":              "footer",
    "Watermark Injection":           "watermark",
    "White Text Injection":          "invisible_unicode",
    "Tiny Font Injection":           "tiny",
    "Invisible Character Injection": "invisible_unicode",
    "Metadata Injection":            "metadata",
}
GENERIC_LOCATIONS = np.array(["header", "footer", "inline"])

print("Insertion strategies registered:", list(INSERTION_STRATEGIES.keys()))
print("Vectorisable locations          :", sorted(VECTORISABLE_LOCATIONS))


Insertion strategies registered: ['header', 'footer', 'watermark', 'metadata', 'inline', 'tiny', 'invisible_unicode']
Vectorisable locations          : ['footer', 'header', 'metadata', 'watermark']


In [ ]:
# ============================================================
# MODULE B -- STEP 4: Load Benign Text Source (frozen, read-only)
# ============================================================
ID_COLUMN_CANDIDATES = [
    "id", "text_id", "doc_id", "document_id", "benign_id",
    "row_id", "sample_id", "uid", "record_id",
]
TEXT_COLUMN_CANDIDATES = [
    "text", "content", "document_text", "body", "benign_text",
    "raw_text", "sentence", "paragraph", "description",
]


def detect_column(df, candidates, what):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    for col in df.columns:
        if any(cand in col.lower() for cand in candidates):
            return col
    print("WARNING: could not auto-detect a {} column among {}".format(what, list(df.columns)))
    return None


benign_df = pd.read_csv(CONFIG["BENIGN_TEXT_CSV"])

ID_COL   = detect_column(benign_df, ID_COLUMN_CANDIDATES,   "id")
TEXT_COL = detect_column(benign_df, TEXT_COLUMN_CANDIDATES, "text")

if TEXT_COL is None:
    string_cols = [c for c in benign_df.columns if benign_df[c].dtype == object]
    if not string_cols:
        raise ValueError("benign_text.csv has no text-like column to inject payloads into.")
    TEXT_COL = string_cols[0]
    print("Falling back to first text-like column: '{}'".format(TEXT_COL))

if ID_COL is None:
    ID_COL = "__source_benign_id__"
    benign_df[ID_COL] = ["BENIGN_TXT_{:06d}".format(i) for i in range(len(benign_df))]
    print("No id column found -- generated stable row-index ids in '{}'.".format(ID_COL))

benign_df = benign_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)

N_BENIGN = len(benign_df)

print("\nBenign text source   :", CONFIG["BENIGN_TEXT_CSV"])
print("Detected id column   :", ID_COL)
print("Detected text column :", TEXT_COL)
print("Usable benign rows   : {:,}".format(N_BENIGN))



Benign text source   : /content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_text.csv
Detected id column   : source_id
Detected text column : extracted_text
Usable benign rows   : 144,937


In [ ]:
# ============================================================
# MODULE B -- STEP 5: Output Validity Detection
#
# Automatically detects whether existing on-disk output follows the
# corrected 1:1 design.  If it does not (old multi-variant strategy,
# wrong row count, duplicate source IDs, outdated pipeline version, or
# schema mismatch), the existing file is marked for full regeneration.
#
# No manual deletion is ever required.
# ============================================================
OUTPUT_COLUMNS = [
    "malicious_id", "source_benign_id", "source_text_hash", "payload_id",
    "attack_family", "attack_name", "attack_location", "attack_type",
    "visibility", "severity", "ocr_survivability", "target_component",
    "label", "generation_timestamp", "malicious_text",
]


def _check_existing_output():
    """
    Returns (existing_df_or_None, is_valid_for_resume: bool, reason: str).

    is_valid_for_resume == True  means: partial or complete 1:1 dataset,
                                        correct schema, current pipeline version.
    is_valid_for_resume == False means: must regenerate from scratch.
    """
    if not os.path.exists(MALICIOUS_TEXT_CSV):
        return None, False, "no_existing_file"

    try:
        ex = pd.read_csv(MALICIOUS_TEXT_CSV)
    except Exception as exc:
        return None, False, "csv_read_error:{}".format(exc)

    # ---- schema check ----
    missing_cols = [c for c in OUTPUT_COLUMNS if c not in ex.columns]
    if missing_cols:
        return None, False, "missing_columns:{}".format(missing_cols)

    # ---- 1:1 design invariant (core correctness check) ----
    n_rows = len(ex)
    if n_rows > N_BENIGN:
        return ex, False, (
            "wrong_row_count:found {:,} rows but benign set has {:,} docs -- "
            "old multi-variant strategy detected (rows > benign)".format(n_rows, N_BENIGN)
        )

    vc = ex["source_benign_id"].value_counts()
    if (vc > 1).any():
        dup_count = int((vc > 1).sum())
        return ex, False, (
            "duplicate_source_ids:{:d} benign docs appear more than once -- "
            "old multi-variant strategy detected".format(dup_count)
        )

    # ---- pipeline version check ----
    if os.path.exists(GENERATION_MANIFEST_JSON):
        try:
            with open(GENERATION_MANIFEST_JSON) as fh:
                manifest = json.load(fh)
            recorded_ver = manifest.get("pipeline_version", "0.0.0")
            rec_parts = [int(x) for x in recorded_ver.split(".")]
            cur_parts = [int(x) for x in PIPELINE_VERSION.split(".")]
            if tuple(rec_parts[:2]) < tuple(cur_parts[:2]):
                return ex, False, (
                    "outdated_pipeline_version:{} < {} -- regenerating with "
                    "corrected strategy".format(recorded_ver, PIPELINE_VERSION)
                )
        except Exception:
            pass   # missing/malformed manifest is not a blocking error for resume

    # ---- duplicate malicious_id check ----
    if ex["malicious_id"].duplicated().any():
        return ex, False, "duplicate_malicious_ids"

    # ---- label integrity ----
    if (ex["label"] != 1).any():
        return ex, False, "non_malicious_labels_found"

    # ---- all checks passed -- resume is safe ----
    return ex, True, "valid_1to1_design"


existing_df_raw, IS_VALID_FOR_RESUME, VALIDITY_REASON = _check_existing_output()

print("=" * 65)
print("OUTPUT VALIDITY CHECK")
print("=" * 65)
if IS_VALID_FOR_RESUME:
    print("  Result  : VALID -- resume mode active")
    print("  Reason  : " + VALIDITY_REASON)
    existing_df = existing_df_raw
    existing_benign_ids = set(existing_df["source_benign_id"].astype(str))
    already_done = len(existing_df)
    remaining = N_BENIGN - already_done
    print("  On disk : {:,} rows already generated.".format(already_done))
    print("  Remaining to generate: {:,}".format(remaining))
else:
    print("  Result  : INVALID -- full regeneration required")
    print("  Reason  : " + VALIDITY_REASON)
    if existing_df_raw is not None:
        print()
        print("  Existing file ({:,} rows) will be overwritten in place.".format(
            len(existing_df_raw)))
        print("  Folder path and filename are preserved (no duplicate files created).")
    else:
        print("  No existing file found -- will generate from scratch.")
    existing_df = pd.DataFrame(columns=OUTPUT_COLUMNS)
    existing_benign_ids = set()
    already_done = 0
print("=" * 65)


OUTPUT VALIDITY CHECK
  Result  : INVALID -- full regeneration required
  Reason  : wrong_row_count:found 856,500 rows but benign set has 144,937 docs -- old multi-variant strategy detected (rows > benign)

  Existing file (856,500 rows) will be overwritten in place.
  Folder path and filename are preserved (no duplicate files created).


In [ ]:
# ============================================================
# MODULE B -- STEP 6: Balanced Stratified Attack-Family Assignment
#
# Fully vectorised with NumPy -- zero Python-level row iteration.
#
# Stage 1 -- Balanced family assignment:
#   A seeded RNG shuffles the family array once.  numpy.tile repeats
#   it until N documents are covered.  Every family receives exactly
#   floor(N/F) or ceil(N/F) documents -- the maximum possible balance.
#
# Stage 2 -- Per-family payload selection:
#   For each family slice, a second seeded RNG cycles through all
#   payloads in permuted order before repeating -- maximises diversity
#   within each family independently.
# ============================================================

# ---- Select only benign docs not yet represented on disk ----
if IS_VALID_FOR_RESUME and existing_benign_ids:
    benign_subset = benign_df[
        ~benign_df[ID_COL].astype(str).isin(existing_benign_ids)
    ].reset_index(drop=True)
    print("Resume mode: {:,} benign docs already done, {:,} remaining.".format(
        len(existing_benign_ids), len(benign_subset)))
else:
    benign_subset = benign_df.copy().reset_index(drop=True)
    print("Fresh generation: {:,} benign documents to process.".format(len(benign_subset)))

N_TO_GENERATE = len(benign_subset)
print("Documents to process this run: {:,}".format(N_TO_GENERATE))

if N_TO_GENERATE == 0:
    print("Nothing to generate -- dataset is already complete.")
    assignment_df = pd.DataFrame()
else:
    # ---- Stage 1: balanced family assignment (vectorised) ----
    rng_families = np.random.default_rng(RANDOM_SEED)
    shuffled_families = rng_families.permutation(FAMILIES_ARR)
    repeats = N_TO_GENERATE // N_FAMILIES + 1
    family_cycle = np.tile(shuffled_families, repeats)[:N_TO_GENERATE]
    benign_subset = benign_subset.copy()
    benign_subset["_assigned_family"] = family_cycle

    # ---- Stage 2: per-family payload selection ----
    # Build payload record list once per family slice (not per row).
    payload_records = []

    for family, fam_benign in benign_subset.groupby("_assigned_family", sort=False):
        fam_payloads = family_to_payloads[family]
        n_fam_docs   = len(fam_benign)
        n_payloads   = len(fam_payloads)

        # Cycle through a seeded permutation of payload indices --
        # each payload appears equally often before any is repeated.
        rng_pay = np.random.default_rng(RANDOM_SEED + abs(hash(family)) % (2**31))
        perm = rng_pay.permutation(n_payloads)
        cycle_idx = np.tile(perm, n_fam_docs // n_payloads + 1)[:n_fam_docs]

        for local_i, (_, benign_row) in enumerate(fam_benign.iterrows()):
            p = fam_payloads.iloc[int(cycle_idx[local_i])]
            payload_records.append({
                "_benign_id":    benign_row[ID_COL],
                "_benign_text":  benign_row[TEXT_COL],
                "_payload_id":   p["payload_id"],
                "_attack_family": family,
                "_attack_name":  p["attack_name"],
                "_payload_text": p["payload_text"],
                "_visibility":   p["visibility"],
                "_severity":     p["severity"],
                "_ocr_surv":     p["ocr_survivability"],
                "_target_comp":  p["target_component"],
            })

    # Convert once -- no incremental concat inside any loop
    assignment_df = pd.DataFrame(payload_records)

    # ---- Resolve attack_location (vectorised map + hash-based fallback) ----
    locations = assignment_df["_attack_family"].map(FAMILY_LOCATION_OVERRIDE)
    mask_generic = locations.isna()
    if mask_generic.any():
        seed_strings = (
            assignment_df.loc[mask_generic, "_attack_family"].astype(str)
            + "|" + assignment_df.loc[mask_generic, "_benign_id"].astype(str)
            + "|" + assignment_df.loc[mask_generic, "_payload_id"].astype(str)
        )
        indices = seed_strings.apply(
            lambda s: int(hashlib.sha1(s.encode()).hexdigest(), 16) % 3
        ).values
        locations[mask_generic] = GENERIC_LOCATIONS[indices]
    assignment_df["_attack_location"] = locations.values

    # ---- Deterministic malicious_id (vectorised SHA-1) ----
    id_strings = (
        assignment_df["_benign_id"].astype(str)
        + "|" + assignment_df["_payload_id"].astype(str)
        + "|" + assignment_df["_attack_location"].astype(str)
    )
    assignment_df["_malicious_id"] = id_strings.apply(
        lambda s: "MAL-TXT-" + hashlib.sha1(s.encode()).hexdigest()[:12]
    )

    # ---- source_text_hash (vectorised) ----
    assignment_df["_src_hash"] = assignment_df["_benign_text"].apply(
        lambda t: hashlib.sha1(t.encode("utf-8", errors="replace")).hexdigest()[:16]
    )

    # ---- Distribution summary ----
    family_dist  = assignment_df["_attack_family"].value_counts()
    location_dist = assignment_df["_attack_location"].value_counts()
    max_fam = int(family_dist.max())

    print("\nAssignment complete ({:,} documents)".format(N_TO_GENERATE))
    print("\nAttack-family distribution:")
    for fam, cnt in family_dist.sort_index().items():
        bar = chr(0x2588) * (cnt * 30 // max_fam)
        print("  {:<32s} {:6,}  {}".format(fam, cnt, bar))
    print("\nAttack-location distribution:")
    for loc, cnt in location_dist.sort_index().items():
        print("  {:<20s} : {:,}".format(loc, cnt))


Fresh generation: 144,937 benign documents to process.
Documents to process this run: 144,937

Assignment complete (144,937 documents)

Attack-family distribution:
  Context Poisoning                 9,663  ██████████████████████████████
  Data Exfiltration                 9,662  █████████████████████████████
  Decision Manipulation             9,662  █████████████████████████████
  Footer Injection                  9,663  ██████████████████████████████
  Header Injection                  9,662  █████████████████████████████
  Hidden Multimodal                 9,662  █████████████████████████████
  Instruction Override              9,663  ██████████████████████████████
  Insurance-specific Attacks        9,663  ██████████████████████████████
  Invisible Character Injection     9,662  █████████████████████████████
  Metadata Injection                9,663  ██████████████████████████████
  OCR-specific Attacks              9,662  █████████████████████████████
  Policy Bypass             

In [ ]:
# ============================================================
# MODULE B -- STEP 7: Vectorised Text Insertion + Checkpointed Save
#
# Pure-string locations (header/footer/watermark/metadata) use pandas
# string operations on the full column slice -- no Python row iteration.
#
# Sentence-parsing locations (inline/tiny/invisible_unicode) use
# .apply() on their sub-slice only (~43% of rows maximum).
#
# Checkpointing writes a partial merged CSV every CHECKPOINT_EVERY rows
# so that an interrupted run loses at most one checkpoint's work.
# ============================================================
CHECKPOINT_EVERY = 10_000   # reduce only if Drive has frequent timeout errors

if N_TO_GENERATE == 0:
    new_rows_df = pd.DataFrame(columns=OUTPUT_COLUMNS)
else:
    GEN_TS = datetime.datetime.now(datetime.timezone.utc).isoformat()

    # ---- Build output DataFrame from assignment columns ----
    out = assignment_df[[
        "_malicious_id", "_benign_id", "_src_hash", "_payload_id",
        "_attack_family", "_attack_name", "_attack_location",
        "_visibility", "_severity", "_ocr_surv", "_target_comp",
        "_benign_text", "_payload_text",
    ]].copy()
    out.columns = [
        "malicious_id", "source_benign_id", "source_text_hash", "payload_id",
        "attack_family", "attack_name", "attack_location",
        "visibility", "severity", "ocr_survivability", "target_component",
        "source_text", "payload_text",
    ]
    out["attack_type"]          = out["attack_location"].map(ATTACK_TYPE_MAP)
    out["label"]                = 1
    out["generation_timestamp"] = GEN_TS
    out["malicious_text"]       = ""   # filled below

    # ---- Vectorised insertion for pure prefix/suffix locations ----
    for loc in VECTORISABLE_LOCATIONS:
        mask = out["attack_location"] == loc
        if not mask.any():
            continue
        src = out.loc[mask, "source_text"]
        pay = out.loc[mask, "payload_text"]
        if loc == "header":
            out.loc[mask, "malicious_text"] = pay + "\n\n" + src
        elif loc == "footer":
            out.loc[mask, "malicious_text"] = src + "\n\n" + pay
        elif loc == "watermark":
            out.loc[mask, "malicious_text"] = (
                src + "\n\n[[WATERMARK:: " + pay + " ::WATERMARK]]"
            )
        elif loc == "metadata":
            out.loc[mask, "malicious_text"] = (
                src + "\n\n<!-- DOCUMENT_METADATA: " + pay + " -->"
            )

    # ---- .apply() for sentence-parsing locations (sub-slices only) ----
    sentence_locs = [
        l for l in out["attack_location"].unique()
        if l not in VECTORISABLE_LOCATIONS
    ]
    for loc in tqdm(sentence_locs, desc="Sentence-parsing locations"):
        mask = out["attack_location"] == loc
        if not mask.any():
            continue
        fn = INSERTION_STRATEGIES[loc]
        out.loc[mask, "malicious_text"] = out.loc[mask].apply(
            lambda r, _fn=fn: _fn(r["source_text"], r["payload_text"]), axis=1
        )

    # ---- Checkpoint support: write partial results to Drive periodically ----
    if CHECKPOINT_EVERY > 0 and N_TO_GENERATE > CHECKPOINT_EVERY:
        for chunk_end in range(CHECKPOINT_EVERY, N_TO_GENERATE, CHECKPOINT_EVERY):
            partial = out.iloc[:chunk_end][OUTPUT_COLUMNS]
            merged_ckpt = pd.concat(
                [existing_df, partial], ignore_index=True
            ).drop_duplicates(subset=["malicious_id"], keep="first")
            _atomic_save_csv(merged_ckpt, MALICIOUS_TEXT_CSV)
            print("  checkpoint: {:,}/{:,} rows written ({:,} total on disk)".format(
                chunk_end, N_TO_GENERATE, len(merged_ckpt)))

    new_rows_df = out[OUTPUT_COLUMNS].copy()
    print("\nText insertion complete. {:,} new rows ready.".format(len(new_rows_df)))


Sentence-parsing locations:   0%|          | 0/3 [00:00<?, ?it/s]

  checkpoint: 10,000/144,937 rows written (10,000 total on disk)
  checkpoint: 20,000/144,937 rows written (20,000 total on disk)
  checkpoint: 30,000/144,937 rows written (30,000 total on disk)
  checkpoint: 40,000/144,937 rows written (40,000 total on disk)
  checkpoint: 50,000/144,937 rows written (50,000 total on disk)
  checkpoint: 60,000/144,937 rows written (60,000 total on disk)
  checkpoint: 70,000/144,937 rows written (70,000 total on disk)
  checkpoint: 80,000/144,937 rows written (80,000 total on disk)
  checkpoint: 90,000/144,937 rows written (90,000 total on disk)
  checkpoint: 100,000/144,937 rows written (100,000 total on disk)
  checkpoint: 110,000/144,937 rows written (110,000 total on disk)
  checkpoint: 120,000/144,937 rows written (120,000 total on disk)
  checkpoint: 130,000/144,937 rows written (130,000 total on disk)
  checkpoint: 140,000/144,937 rows written (140,000 total on disk)

Text insertion complete. 144,937 new rows ready.


In [ ]:
# ============================================================
# MODULE B -- STEP 8: Merge, Dedup Safety Net, Atomic Save (in place)
# ============================================================
if N_TO_GENERATE == 0:
    final_df    = existing_df.copy()
    save_status = "skipped_already_complete"
else:
    final_df = pd.concat([existing_df, new_rows_df], ignore_index=True)

    before_dedup = len(final_df)
    final_df = final_df.drop_duplicates(
        subset=["malicious_id"], keep="first"
    ).reset_index(drop=True)
    if len(final_df) != before_dedup:
        print("Safety-net dedup removed {:,} duplicate row(s).".format(
            before_dedup - len(final_df)))

    _atomic_save_csv(final_df, MALICIOUS_TEXT_CSV)

    if (not IS_VALID_FOR_RESUME) and (existing_df_raw is not None) and len(existing_df_raw) > 0:
        save_status = "overwritten_in_place"
    else:
        save_status = "written"

    # Clean up any checkpoint .tmp files
    _cleanup_tmp_files()

    # ---- lightweight meta sidecar ----
    _atomic_save_json({
        "rows":               len(final_df),
        "pipeline_version":   PIPELINE_VERSION,
        "attack_families":    int(final_df["attack_family"].nunique()),
        "generation_strategy": "balanced_stratified_1to1",
        "new_rows_this_run":  int(len(new_rows_df)),
        "updated_at":         datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "generator":          "MODULE_B",
    }, MALICIOUS_TEXT_META_JSON)

print("Save status    :", save_status)
print("Output file    :", MALICIOUS_TEXT_CSV)
print("Total rows     : {:,}".format(len(final_df)))
print("Expected rows  : {:,}  (one per benign document)".format(N_BENIGN))
if len(final_df) != N_BENIGN:
    print("WARNING: row count mismatch -- {:,} docs unaccounted for.".format(
        N_BENIGN - len(final_df)))


Save status    : overwritten_in_place
Output file    : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/malicious_text.csv
Total rows     : 144,937
Expected rows  : 144,937  (one per benign document)


In [ ]:
# ============================================================
# MODULE B -- STEP 9: Generation Manifest (generation_manifest.json)
#
# Stores all metadata needed for reproducibility auditing, dataset
# version tracking, and compatibility checks by future modules.
# ============================================================
_ref_df = final_df if len(final_df) > 0 else existing_df

def _vcounts(series):
    return {k: int(v) for k, v in series.value_counts().items()}

manifest = {
    "dataset_version":                DATASET_VERSION,
    "pipeline_version":               PIPELINE_VERSION,
    "generation_strategy":            "balanced_stratified_1to1",
    "generation_timestamp":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "random_seed":                    RANDOM_SEED,
    "runtime_ocr_enabled":            CONFIG["RUNTIME_OCR_ONLY"],
    "number_of_legitimate_documents": N_BENIGN,
    "number_of_generated_documents":  len(final_df),
    "1to1_mapping_verified":          len(final_df) == N_BENIGN,
    "attack_family_distribution":     _vcounts(_ref_df["attack_family"]),
    "payload_distribution":           _vcounts(_ref_df["payload_id"]),
    "dataset_statistics": {
        "attack_location_distribution":    _vcounts(_ref_df["attack_location"]),
        "severity_distribution":           _vcounts(_ref_df["severity"]),
        "visibility_distribution":         _vcounts(_ref_df["visibility"]),
        "ocr_survivability_distribution":  _vcounts(_ref_df["ocr_survivability"]),
        "unique_source_benign_ids":        int(_ref_df["source_benign_id"].nunique()),
        "unique_payloads_used":            int(_ref_df["payload_id"].nunique()),
        "attack_type_distribution":        _vcounts(_ref_df["attack_type"]),
    },
    "output_file":       MALICIOUS_TEXT_CSV,
    "benign_source_file": CONFIG["BENIGN_TEXT_CSV"],
    "payload_bank_file": PAYLOAD_BANK_CSV,
}

_atomic_save_json(manifest, GENERATION_MANIFEST_JSON)

# Print manifest without the verbose payload_distribution field
_display = {k: v for k, v in manifest.items() if k != "payload_distribution"}
print("Generation manifest saved:", GENERATION_MANIFEST_JSON)
print(json.dumps(_display, indent=2))


Generation manifest saved: /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/generation_manifest.json
{
  "dataset_version": "1.0",
  "pipeline_version": "2.1.0",
  "generation_strategy": "balanced_stratified_1to1",
  "generation_timestamp": "2026-06-22T08:11:23.264504+00:00",
  "random_seed": 42,
  "runtime_ocr_enabled": true,
  "number_of_legitimate_documents": 144937,
  "number_of_generated_documents": 144937,
  "1to1_mapping_verified": true,
  "attack_family_distribution": {
    "Instruction Override": 9663,
    "Insurance-specific Attacks": 9663,
    "Metadata Injection": 9663,
    "Footer Injection": 9663,
    "Context Poisoning": 9663,
    "Policy Bypass": 9663,
    "White Text Injection": 9663,
    "OCR-specific Attacks": 9662,
    "Hidden Multimodal": 9662,
    "Decision Manipulation": 9662,
    "Header Injection": 9662,
    "Tiny Font Injection": 9662,
    "Data Exfiltration": 9662,
    "Watermark Injection": 9662,
    "Invisible Character Injection": 9662
  },
  "dataset_

In [ ]:
# ============================================================
# MODULE B -- STEP 10: Full Dataset Validation
#
# Validates: schema, null values, 1:1 design invariant, coverage of
# every benign doc, duplicate IDs, label integrity, controlled
# vocabulary, payload traceability, attack-family balance.
# ============================================================
ALLOWED_ATTACK_LOCATIONS = set(INSERTION_STRATEGIES.keys())
ALLOWED_ATTACK_TYPES     = set(ATTACK_TYPE_MAP.values())
ALLOWED_SEVERITY         = {"low", "medium", "high", "critical"}
ALLOWED_VISIBILITY       = {"visible", "hidden", "invisible"}


def validate_malicious_text(df, benign_ids_set):
    report = {"errors": [], "warnings": [], "stats": {}}

    # ---- schema ----
    missing_cols = [c for c in OUTPUT_COLUMNS if c not in df.columns]
    if missing_cols:
        report["errors"].append("Missing required columns: " + str(missing_cols))
        report["is_valid"] = False
        return report

    # ---- null / empty values ----
    for col in OUTPUT_COLUMNS:
        n_null = int(df[col].isna().sum())
        if n_null:
            report["errors"].append(
                "Column '{}' has {:,} null value(s)".format(col, n_null))
    empty_text = int((df["malicious_text"].astype(str).str.strip() == "").sum())
    if empty_text:
        report["errors"].append(
            "{:,} row(s) have empty malicious_text".format(empty_text))

    # ---- 1:1 design invariant ----
    if len(df) != N_BENIGN:
        report["errors"].append(
            "Row count {:,} != expected {:,} (one per benign document)".format(
                len(df), N_BENIGN))
    dup_src = int(df["source_benign_id"].duplicated().sum())
    if dup_src:
        report["errors"].append(
            "{:,} duplicate source_benign_id value(s) -- 1:1 invariant violated".format(dup_src))

    # ---- complete coverage: every benign id must have a malicious counterpart ----
    generated_ids = set(df["source_benign_id"].astype(str))
    missing_ids   = benign_ids_set - generated_ids
    if missing_ids:
        report["errors"].append(
            "{:,} benign document(s) have no malicious counterpart".format(len(missing_ids)))

    # ---- duplicate malicious_id ----
    dup_mal = int(df["malicious_id"].duplicated().sum())
    if dup_mal:
        report["errors"].append(
            "{:,} duplicate malicious_id value(s)".format(dup_mal))

    # ---- label integrity ----
    bad_labels = int((df["label"] != 1).sum())
    if bad_labels:
        report["errors"].append(
            "{:,} row(s) do not have label == 1".format(bad_labels))

    # ---- controlled vocabulary ----
    bad_loc = df[~df["attack_location"].isin(ALLOWED_ATTACK_LOCATIONS)]
    if len(bad_loc):
        report["errors"].append(
            "Invalid attack_location values: " + str(bad_loc["attack_location"].unique().tolist()))
    bad_type = df[~df["attack_type"].isin(ALLOWED_ATTACK_TYPES)]
    if len(bad_type):
        report["errors"].append(
            "Invalid attack_type values: " + str(bad_type["attack_type"].unique().tolist()))
    bad_sev = df[~df["severity"].isin(ALLOWED_SEVERITY)]
    if len(bad_sev):
        report["warnings"].append(
            "Unexpected severity values: " + str(bad_sev["severity"].unique().tolist()))
    bad_vis = df[~df["visibility"].isin(ALLOWED_VISIBILITY)]
    if len(bad_vis):
        report["warnings"].append(
            "Unexpected visibility values: " + str(bad_vis["visibility"].unique().tolist()))

    # ---- payload traceability ----
    valid_payload_ids = set(payload_bank_df["payload_id"])
    orphan = df[~df["payload_id"].isin(valid_payload_ids)]
    if len(orphan):
        report["warnings"].append(
            "{:,} row(s) reference a payload_id not in the current payload bank".format(len(orphan)))

    # ---- attack-family balance (max - min <= 1 for perfect balance) ----
    fam_counts = df["attack_family"].value_counts()
    if len(fam_counts) > 0:
        max_c, min_c = int(fam_counts.max()), int(fam_counts.min())
        if max_c - min_c > 1:
            report["warnings"].append(
                "Attack-family imbalance: max={:,}, min={:,} "
                "(difference > 1 is unexpected)".format(max_c, min_c))

    # ---- stats ----
    report["stats"] = {
        "total_malicious_samples":        len(df),
        "attack_families":                int(df["attack_family"].nunique()),
        "unique_payloads_used":           int(df["payload_id"].nunique()),
        "unique_source_benign_ids":       int(df["source_benign_id"].nunique()),
        "samples_per_family":             df["attack_family"].value_counts().to_dict(),
        "samples_per_location":           df["attack_location"].value_counts().to_dict(),
        "severity_distribution":          df["severity"].value_counts().to_dict(),
        "visibility_distribution":        df["visibility"].value_counts().to_dict(),
        "ocr_survivability_distribution": df["ocr_survivability"].value_counts().to_dict(),
    }
    report["is_valid"] = len(report["errors"]) == 0
    return report


benign_ids_set    = set(benign_df[ID_COL].astype(str))
validation_report = validate_malicious_text(final_df, benign_ids_set)

print("=" * 70)
print("MODULE B VALIDATION REPORT")
print("=" * 70)
print("Status  :", "PASSED" if validation_report["is_valid"] else "FAILED")

if validation_report["errors"]:
    print("\nErrors:")
    for e in validation_report["errors"]:
        print("  x", e)

if validation_report["warnings"]:
    print("\nWarnings:")
    for w in validation_report["warnings"]:
        print("  !", w)

print("\nStats:")
for k, v in validation_report["stats"].items():
    if isinstance(v, dict):
        print("  {}:".format(k))
        for kk, vv in sorted(v.items()):
            print("    {}: {:,}".format(kk, vv) if isinstance(vv, int) else
                  "    {}: {}".format(kk, vv))
    else:
        print("  {}: {:,}".format(k, v) if isinstance(v, int) else
              "  {}: {}".format(k, v))


MODULE B VALIDATION REPORT
Status  : PASSED

Stats:
  total_malicious_samples: 144,937
  attack_families: 15
  unique_payloads_used: 58
  unique_source_benign_ids: 144,937
  samples_per_family:
    Context Poisoning: 9,663
    Data Exfiltration: 9,662
    Decision Manipulation: 9,662
    Footer Injection: 9,663
    Header Injection: 9,662
    Hidden Multimodal: 9,662
    Instruction Override: 9,663
    Insurance-specific Attacks: 9,663
    Invisible Character Injection: 9,662
    Metadata Injection: 9,663
    OCR-specific Attacks: 9,662
    Policy Bypass: 9,663
    Tiny Font Injection: 9,662
    Watermark Injection: 9,662
    White Text Injection: 9,663
  samples_per_location:
    footer: 35,491
    header: 35,340
    inline: 25,794
    invisible_unicode: 19,325
    metadata: 9,663
    tiny: 9,662
    watermark: 9,662
  severity_distribution:
    critical: 33,014
    high: 81,808
    medium: 30,115
  visibility_distribution:
    hidden: 46,700
    invisible: 19,325
    visible: 78,912


In [ ]:
# ============================================================
# MODULE B -- FINAL SUMMARY
# ============================================================
print("=" * 70)
print("MODULE B SUMMARY (pipeline v{})".format(PIPELINE_VERSION))
print("=" * 70)
print("Project root           :", CONFIG["PROJECT_ROOT"])
print("Payload bank   (in)    :", PAYLOAD_BANK_CSV)
print("Benign text    (in)    :", CONFIG["BENIGN_TEXT_CSV"])
print("Malicious text (out)   :", MALICIOUS_TEXT_CSV)
print("Manifest       (out)   :", GENERATION_MANIFEST_JSON)
print("Generation strategy    : balanced_stratified_1to1")
print("Random seed            :", RANDOM_SEED)
print("Attack families used   :", N_FAMILIES)
print("Benign documents       : {:,}".format(N_BENIGN))
print("Malicious documents    : {:,}".format(len(final_df)))
print("1:1 mapping preserved  :", len(final_df) == N_BENIGN)
print("New rows this run      : {:,}".format(N_TO_GENERATE))
print("Save action taken      :", save_status)
print("Validity at start      :", VALIDITY_REASON)
print("Validation status      :", "PASSED" if validation_report["is_valid"] else "FAILED")
print("Runtime-OCR-only       :", CONFIG["RUNTIME_OCR_ONLY"],
      " (no OCR performed in Module B)")
print("Resume enabled         :", CONFIG["RESUME_ENABLED"])
print("=" * 70)

print("MODULE B COMPLETED")


MODULE B SUMMARY (pipeline v2.1.0)
Project root           : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET
Payload bank   (in)    : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PAYLOAD_BANK/insurance_payload_bank.csv
Benign text    (in)    : /content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_text.csv
Malicious text (out)   : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/malicious_text.csv
Manifest       (out)   : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/generation_manifest.json
Generation strategy    : balanced_stratified_1to1
Random seed            : 42
Attack families used   : 15
Benign documents       : 144,937
Malicious documents    : 144,937
1:1 mapping preserved  : True
New rows this run      : 144,937
Save action taken      : overwritten_in_place
Validity at start      : wrong_row_count:found 856,500 rows but benign set has 144,937 docs -- old multi-variant strategy detected (rows > benign)
Validation status      : PASSED
Runtime-OCR-only    

In [ ]:
from pathlib import Path

for f in Path("/content/drive/MyDrive").rglob("generation_manifest.json"):
    print(f)

/content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/generation_manifest.json
